# ⚠️ DEPRECATED NOTEBOOK
This notebook is superseded by **notebooks/01_data_exploration.ipynb**, which is the authoritative NHANES pipeline for GuidaPlate.

**Do not run this notebook.** It was retained for reference only. If executed, it will raise an error because `CKD_PATIENTS_CSV` was removed from `backend/config.py` during final cleanup — this notebook's output (`ckd_patients.csv`) is no longer used anywhere in the canonical pipeline (use `ckd_cohort_final.csv` from notebook 01 instead); this failure is intentional, not a bug.

The canonical CKD cohort (1,862 patients) and all downstream outputs are produced by notebook 01.

**To reproduce results, run notebooks in this order:**
1. `01_data_exploration.ipynb` → produces `data/processed/ckd_cohort_final.csv`
2. `03_statistical_analysis.ipynb` → produces stats and figures
3. `04_xgboost_training.ipynb` → produces `models/xgboost_v1.pkl`
4. `05_lstm_training.ipynb` → produces `models/lstm_final.keras`
5. `06_evaluation.ipynb` → produces SHAP figures and McNemar results

# NHANES CKD Cohort Building Pipeline

This notebook loads NHANES **CSV** extracts (J cycle), merges participant records, computes **eGFR (CKD-EPI 2021)** and **KDIGO G-stages**, filters to **CKD (eGFR < 90)**, selects dietary and clinical columns, and saves `ckd_patients.csv`.


## Section 1 — Setup and imports

- Import libraries
- Load paths from `config.py`
- Print versions


In [ ]:

import sys
from pathlib import Path

def project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "config.py").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Run from the GUIDAPLATE repo (config.py not found).")

ROOT = project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "app") not in sys.path:
    sys.path.insert(0, str(ROOT / "app"))

import config
import utils

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("ROOT:", ROOT)
print("NHANES CSV dir:", config.NHANES_DIR)


## Section 2 — Load raw NHANES files

Load Day 1 / Day 2 dietary totals, standard biochemistry (creatinine), and demographics.


In [ ]:

import sys
from pathlib import Path

def project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "config.py").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Run from the GUIDAPLATE repo (config.py not found).")

ROOT = project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "app") not in sys.path:
    sys.path.insert(0, str(ROOT / "app"))

import config
import utils

import pandas as pd

dr1 = utils.load_csv(config.CSV_DR1TOT, low_memory=False)
dr2 = utils.load_csv(config.CSV_DR2TOT, low_memory=False)
bio = utils.load_csv(config.CSV_BIOPRO, low_memory=False)
demo = utils.load_csv(config.CSV_DEMO, low_memory=False)

from IPython.display import display

for name, df in [("DR1TOT_J", dr1), ("DR2TOT_J", dr2), ("BIOPRO_J", bio), ("DEMO_J", demo)]:
    print(f"{name} shape: {df.shape}")
    display(df.head(3))


## Section 3 — Merge all files on `SEQN`

Inner join **demographics + labs** (needed for eGFR), then **left** join dietary totals.


In [ ]:

import sys
from pathlib import Path

def project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "config.py").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Run from the GUIDAPLATE repo (config.py not found).")

ROOT = project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "app") not in sys.path:
    sys.path.insert(0, str(ROOT / "app"))

import config
import utils

import pandas as pd

# Use inner join for participants present in both DEMO and BIOPRO (creatinine + age/sex)
base = demo.merge(bio, on="SEQN", how="inner", suffixes=("_demo", "_bio"))
print("After DEMO ∩ BIOPRO:", base.shape)

merged = base.merge(dr1, on="SEQN", how="left").merge(dr2, on="SEQN", how="left")
print("After +DR1 +DR2 (left joins):", merged.shape)

missing = merged.isna().sum().sort_values(ascending=False)
print("Top 15 missing-value counts:")
print(missing.head(15).to_string())


## Section 4 — Calculate eGFR and CKD stage

- **eGFR**: CKD-EPI 2021 creatinine-only (`utils.egfr_ckd_epi_2021_creatinine`)
- **Stages**: G1–G5 from eGFR cutoffs
- Bar chart of stage distribution (before filtering)


In [ ]:

import sys
from pathlib import Path

def project_root() -> Path:
    p = Path.cwd().resolve()
    for _ in range(8):
        if (p / "config.py").exists():
            return p
        p = p.parent
    raise FileNotFoundError("Run from the GUIDAPLATE repo (config.py not found).")

ROOT = project_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(ROOT / "app") not in sys.path:
    sys.path.insert(0, str(ROOT / "app"))

import config
import utils

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

scr = pd.to_numeric(merged["LBXSCR"], errors="coerce")
age = pd.to_numeric(merged["RIDAGEYR"], errors="coerce")
sex = pd.to_numeric(merged["RIAGENDR"], errors="coerce")

merged["eGFR"] = utils.egfr_ckd_epi_2021_creatinine(scr_mg_dl=scr, age_years=age, sex=sex)

e = pd.to_numeric(merged["eGFR"], errors="coerce")
stage = pd.Series(pd.NA, index=merged.index, dtype="string")
stage[e.ge(90)] = "G1"
stage[e.between(60, 89.999999, inclusive="both")] = "G2"
stage[e.between(45, 59.999999, inclusive="both")] = "G3a"
stage[e.between(30, 44.999999, inclusive="both")] = "G3b"
stage[e.between(15, 29.999999, inclusive="both")] = "G4"
stage[e.lt(15)] = "G5"
merged["CKD_stage"] = stage

print("Stage distribution (all merged participants with valid eGFR):")
vc = merged["CKD_stage"].value_counts(dropna=False)
print(vc.to_string())

plt.figure(figsize=(8, 4))
merged["CKD_stage"].astype(str).value_counts().reindex(["G1","G2","G3a","G3b","G4","G5"]).fillna(0).plot(kind="bar", color="steelblue")
plt.title("CKD stage distribution (pre-filter)")
plt.xlabel("Stage")
plt.ylabel("Count")
plt.tight_layout()
plt.show()


## Section 5 — Filter CKD patients

Keep **eGFR < 90** (excludes G1 / normal kidney function for this cohort file).


In [ ]:

import pandas as pd

before = len(merged)
ckd = merged.loc[pd.to_numeric(merged["eGFR"], errors="coerce").lt(90)].copy()
after = len(ckd)
print("Patients before filter:", before)
print("Patients kept (eGFR < 90):", after)
print("Patients removed:", before - after)


## Section 6 — Select and rename columns

Keep clinical identifiers, eGFR stage, energy, protein/sodium, and potassium/phosphorus for both recall days.


In [ ]:

cols = [
    "SEQN", "RIDAGEYR", "RIAGENDR", "LBXSCR", "eGFR", "CKD_stage",
    "DR1TKCAL", "DR1TPROT", "DR1TSODI", "DR1TPOTA", "DR1TPHOS",
    "DR2TKCAL", "DR2TPROT", "DR2TSODI", "DR2TPOTA", "DR2TPHOS",
]
missing_cols = [c for c in cols if c not in ckd.columns]
if missing_cols:
    raise KeyError(f"Missing expected columns: {missing_cols}")

ckd_out = ckd[cols].copy()
print("Final shape:", ckd_out.shape)
print("Columns:", list(ckd_out.columns))


## Section 7 — Save output


In [ ]:

ckd_out.to_csv(config.CKD_PATIENTS_CSV, index=False)
print("Saved:", config.CKD_PATIENTS_CSV)
print("Confirmation shape:", ckd_out.shape)
